In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [32]:
!head ratings_long.csv

userId,movieId,rating
0,16,5
0,72,5
0,86,5
0,259,1
0,319,4
0,521,4
0,534,2
0,671,1
0,673,2


In [33]:
r = np.full((20, 1000),fill_value=np.nan)

In [34]:
df = pd.read_csv('ratings_long.csv')

In [35]:
for rec in df.itertuples():
    r[rec.userId][rec.movieId] = rec.rating

Note that $r$ matrix is $20 \times 1000$ with only <1\% full (highly sparse)

In [36]:
r

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan,  4., nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]])

What if we define two matricies
- u = $20 \times 4$
- v = $4 \times 1000$


Then model $r$ as $u \times v$

Problem is we have to learn for $20 \times 4 + 4 \times 1000 = 4080$ parameters (better than 20.000 x 0.99 missing values)

## Problem

1. Define a convex loss function wrt $u$ and $v$
- Solve using gradient descent algorithm explained in **I Do**
- Use any regulatizer $L1$ or $L2$ to prevent overfitting

In [39]:

- Paramları belirleme
- np randomla rastgele u ve v doldurur
- Loss ve error hesaplanır, L2 kullandım çünkü feature az ve L1 featureları 0a çeker 
- Klasik prediction ve gradient hesaplama döngüsü çalışır
- En son matrix reconstruct edilir.

SyntaxError: invalid decimal literal (992343531.py, line 3)

In [38]:
LATENT_DIM = 4
LEARNING_RATE = 0.01
REGULARIZATION = 0.05
EPOCHS = 5000
PRINT_EVERY = 250
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

mask = ~np.isnan(r)

num_users, num_movies = r.shape

U = np.random.normal(0, 0.1, (num_users, LATENT_DIM))
V = np.random.normal(0, 0.1, (LATENT_DIM, num_movies))


def compute_error(ratings, prediction, observed_mask):
    error = np.zeros_like(prediction)
    error[observed_mask] = prediction[observed_mask] - ratings[observed_mask]
    return error


def compute_loss(error, u, v):
    mse = 0.5 * np.sum(error[mask] ** 2)
    reg = 0.5 * REGULARIZATION * (np.sum(u ** 2) + np.sum(v ** 2))
    return mse + reg


losses = []

for epoch in range(EPOCHS):

    prediction = U @ V

    error = compute_error(r, prediction, mask)

    grad_u = error @ V.T + REGULARIZATION * U
    grad_v = U.T @ error + REGULARIZATION * V

    U -= LEARNING_RATE * grad_u
    V -= LEARNING_RATE * grad_v

    loss = compute_loss(error, U, V)
    losses.append(loss)

    if epoch % PRINT_EVERY == 0 or epoch == EPOCHS - 1:
        print(f"Epoch {epoch:4d}/{EPOCHS}   Loss = {loss:.4f}")

# Reconstruced matrix
R_hat = U @ V

# Training RMSE hesabı
rmse = np.sqrt(np.mean((R_hat[mask] - r[mask]) ** 2))

print("Training Finished")
print(f"Final Loss : {losses[-1]:.4f}")
print(f"Training RMSE : {rmse:.4f}")

Epoch    0/5000   Loss = 1108.3618
Epoch  250/5000   Loss = 10.8478
Epoch  500/5000   Loss = 10.6740
Epoch  750/5000   Loss = 10.5373
Epoch 1000/5000   Loss = 10.4293
Epoch 1250/5000   Loss = 10.3439
Epoch 1500/5000   Loss = 10.2760
Epoch 1750/5000   Loss = 10.2218
Epoch 2000/5000   Loss = 10.1785
Epoch 2250/5000   Loss = 10.1436
Epoch 2500/5000   Loss = 10.1154
Epoch 2750/5000   Loss = 10.0924
Epoch 3000/5000   Loss = 10.0735
Epoch 3250/5000   Loss = 10.0578
Epoch 3500/5000   Loss = 10.0448
Epoch 3750/5000   Loss = 10.0338
Epoch 4000/5000   Loss = 10.0245
Epoch 4250/5000   Loss = 10.0164
Epoch 4500/5000   Loss = 10.0094
Epoch 4750/5000   Loss = 10.0033
Epoch 4999/5000   Loss = 9.9979
Training Finished
Final Loss : 9.9979
Training RMSE : 0.0157


In [ ]:
Örnek tahminler ve reconstruced matrix printi

In [30]:
# Örnek tahminler

print("\nÖrnek Predictionlar")
print("-" * 60)

observed = np.argwhere(mask)

for idx in range(min(20, len(observed))):
    user, movie = observed[idx]
    actual = r[user, movie]
    predicted = R_hat[user, movie]

    print(
        f"User {user:2d} | "
        f"Movie {movie:4d} | "
        f"Actual = {actual:.1f} | "
        f"Predicted = {predicted:.2f}"
    )

# Reconstructed matrixin bir kısmını göster

print("\nİlk 5 User x İlk 10 Film (Predicted Ratingler)")
print(np.round(R_hat[:5, :10], 2))


Sample Predictions
------------------------------------------------------------
User  0 | Movie   16 | Actual = 5.0 | Predicted = 4.99
User  0 | Movie   72 | Actual = 5.0 | Predicted = 4.98
User  0 | Movie   86 | Actual = 5.0 | Predicted = 4.98
User  0 | Movie  259 | Actual = 1.0 | Predicted = 1.00
User  0 | Movie  319 | Actual = 4.0 | Predicted = 3.98
User  0 | Movie  521 | Actual = 4.0 | Predicted = 3.98
User  0 | Movie  534 | Actual = 2.0 | Predicted = 1.99
User  0 | Movie  671 | Actual = 1.0 | Predicted = 1.00
User  0 | Movie  673 | Actual = 2.0 | Predicted = 1.99
User  0 | Movie  739 | Actual = 3.0 | Predicted = 2.98
User  0 | Movie  764 | Actual = 2.0 | Predicted = 1.99
User  0 | Movie  798 | Actual = 3.0 | Predicted = 2.99
User  0 | Movie  917 | Actual = 4.0 | Predicted = 3.98
User  1 | Movie   87 | Actual = 1.0 | Predicted = 1.00
User  1 | Movie  125 | Actual = 5.0 | Predicted = 4.98
User  1 | Movie  129 | Actual = 1.0 | Predicted = 1.00
User  1 | Movie  148 | Actual = 1.0 | P